# 10 — Design power and protocol lock

**Objective.** Run development-only operating-characteristic simulations, freeze final budgets/losses/candidate gates, and issue the second HMAC protocol lock.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Notebook 11 will refuse final-test execution unless this lock is final-grade and satisfies the configured HMAC requirement.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("10", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import json
import numpy as np
import pandas as pd
import yaml
from cruxvc.hashing import sha256_file
from cruxvc.io import read_json, read_table, write_json, write_table
from cruxvc.manifest import create_protocol_lock, signing_key_from_environment, verify_protocol_lock
from cruxvc.power import simulate_rq1_power
from cruxvc.selective import build_frozen_candidate_family

signing_key = signing_key_from_environment()
verify_protocol_lock(P.locks / "phase0_lock.json", signing_key=signing_key, require_hmac=False)
pilot = read_table(P.attributions / "development_pilot_attributions.parquet")
diagnostics = pd.read_csv(P.audits / "09_approximation_repeat_diagnostics.csv")
runtime = pd.read_csv(P.audits / "09_explanation_runtime.csv")
gate_scores = read_table(P.selective / "development_gate_scores_2008.parquet").set_index("case_id")
pilot_losses = read_table(P.selective / "development_explanation_losses_2008.parquet")
faithfulness_scale_payload = read_json(P.protocol / "development_faithfulness_scale.json")
statistical = yaml.safe_load((P.config / "statistical_analysis.yaml").read_text())

In [ ]:
# Pilot-driven noise parameters; these are inputs to, not substitutes for, the final paired design.
cross_sd = float(pilot.groupby(["case_id", "feature_group"])["phi_log_odds"].mean().groupby("case_id").std().mean())
within_sd = float(diagnostics["approximation_sd"].fillna(0).std())
cross_sd = max(cross_sd, 0.05)
within_sd = max(within_sd, 0.01)
power = simulate_rq1_power(
    n_cases=int(PROFILE["local_audit_n"]),
    cross_sd=cross_sd,
    within_sd=within_sd,
    effect_grid=statistical["power"]["effect_grid"],
    refit_grid=statistical["power"]["refit_grid"],
    repetitions=int(PROFILE["power_simulations"]),
    meaningful_effect=float(statistical["smallest_meaningful_effects"]["rq1_delta_spec"]),
    equivalence_half_width=float(statistical["power"]["equivalence_half_width"]),
    seed=int(CFG["execution"]["random_seed"]) + 100,
)
power_path = write_table(power, P.audits / "10_rq1_power_operating_characteristics.csv")

In [ ]:
orientations = {column: True for column in gate_scores.columns}
candidates = build_frozen_candidate_family(
    gate_scores,
    CFG["selective_risk"]["candidate_acceptance_grid"],
    orientations=orientations,
)
if len(candidates) > 90:
    # Preserve the protocol's finite-family ceiling by deterministic score priority.
    score_priority = ["confidence_loss", "margin_loss", "entropy", "epistemic_variance", "model_disagreement", "density_ood", "predicted_explanation_loss"]
    candidates["score_priority"] = candidates["score_name"].map({name: i for i, name in enumerate(score_priority)}).fillna(99)
    candidates = candidates.sort_values(["score_priority", "candidate_id"]).head(90).drop(columns="score_priority")
candidate_path = write_table(candidates, P.selective / "frozen_candidate_policies.parquet")

losses = {
    "L_S": "mean sqrt-JSD to a finite frozen same-outcome reference panel",
    "L_C": "mean confirmatory construct contrast sqrt-JSD",
    "L_F": "one minus clipped development-scaled top-minus-random conditional AOPC",
    "L_Y": "0/1 prediction error",
    "missing_or_failed_explanation": 1.0,
}
loss_path = write_json(losses, P.protocol / "frozen_explanation_losses.json")

In [ ]:
final_grade = bool(PROFILE["is_final"])
if final_grade and CFG["execution"]["require_hmac_for_final_test"] and not signing_key:
    raise RuntimeError("Full-profile final lock requires CRUX_PROTOCOL_SIGNING_KEY in the environment or Colab secrets")
design = {
    "lock_type": "design",
    "protocol_version": "2.2",
    "final_grade": final_grade,
    "compute_profile": PROFILE,
    "phase0_lock_sha256": sha256_file(P.locks / "phase0_lock.json"),
    "pilot_attributions_sha256": sha256_file(P.attributions / "development_pilot_attributions.parquet"),
    "pilot_explanation_losses_sha256": sha256_file(P.selective / "development_explanation_losses_2008.parquet"),
    "development_faithfulness_scale_sha256": sha256_file(P.protocol / "development_faithfulness_scale.json"),
    "runtime_benchmark_sha256": sha256_file(P.audits / "09_explanation_runtime.csv"),
    "power_curve_sha256": sha256_file(power_path),
    "candidate_family_sha256": sha256_file(candidate_path),
    "loss_definitions_sha256": sha256_file(loss_path),
    "risk_budgets": {
        "rho_S": float(CFG["selective_risk"]["rho_S"]),
        "rho_C": float(CFG["selective_risk"]["rho_C"]),
        "rho_F": float(CFG["selective_risk"]["rho_F"]),
        "rho_Y": float(CFG["selective_risk"]["rho_Y"]),
        "q_min": float(CFG["selective_risk"]["q_min"]),
        "familywise_delta": float(CFG["selective_risk"]["familywise_delta"]),
        "accepted_calibration_min": int(CFG["selective_risk"]["accepted_calibration_min"]),
    },
    "explanation_budgets": {
        "background_sets": int(PROFILE["background_sets"]),
        "background_n": int(PROFILE["background_n"]),
        "local_audit_n": int(PROFILE["local_audit_n"]),
        "permutation_orderings": int(PROFILE["permutation_orderings"]),
        "approximation_seeds": int(PROFILE["approximation_seeds"]),
        "bootstrap_refits": int(PROFILE["bootstrap_refits"]),
    },
    "esrc_operational_panel": {
        "prediction_outcome": str(CFG["selective_risk"]["prediction_outcome"]),
        "reference_panel_refits": int(PROFILE["esrc_reference_panel_refits"]),
        "permutation_orderings": int(PROFILE["esrc_permutation_orderings"]),
        "faithfulness_random_repetitions": int(PROFILE["esrc_faithfulness_random_repetitions"]),
        "faithfulness_scale": float(faithfulness_scale_payload["value"]),
        "faithfulness_scale_definition": str(faithfulness_scale_payload["definition"]),
        "missing_or_failed_explanation_loss": 1.0,
    },
    "pilot_cross_sd": cross_sd,
    "pilot_within_sd": within_sd,
    "pilot_loss_summary": {
        column: float(pilot_losses[column].mean())
        for column in ["L_S", "L_C", "L_F"]
    },
    "test_results_opened": False,
}
design_lock = create_protocol_lock(
    design,
    P.locks / "design_lock.json",
    author_email=CFG["project"]["author_email"],
    signing_key=signing_key,
)
design_snapshot = write_json(design, P.protocol / "design_snapshot.json")

In [ ]:
CTX.recorder.complete([power_path, candidate_path, loss_path, design_lock, design_snapshot], extra={"final_grade": final_grade})
print(f"Design lock created: final_grade={final_grade}, profile={PROFILE['name']}, candidates={len(candidates)}")